# DENTRACE: YOLOv8n-Seg Training on DENTEX Dental Panoramic Radiographs

**Project:** DENTRACE (Local-First Dental Panoramic Anomaly Detection)
**Model Architecture:** YOLOv8n-seg (Transfer Fine-Tuned from `yolov8n-seg.pt`)
**Dataset:** DENTEX Challenge 2023 (Abnormal Tooth Detection & Segmentation)
**Pathology Classes (4):** `0: impacted_tooth`, `1: caries`, `2: periapical_lesion`, `3: deep_caries`
**Hardware Target:** Google Colab T4 GPU (16 GB VRAM)
**Clinical Governance Boundary:** Academic Decision Support. *Draft for dentist review. Not a diagnosis.*
**License:** CC BY-NC 4.0 (DENTEX Academic Terms)

## 1. Environment Setup & Hardware Verification
Verify NVIDIA T4 GPU availability and install required dependencies.

In [ ]:
# Check GPU acceleration
!nvidia-smi

# Install Ultralytics and export toolchains
!pip install --upgrade ultralytics onnx onnxruntime onnxslim albumentations pydantic

## 2. Dataset Preparation (DENTEX YOLOv8-Seg Format)
This cell creates the training and validation directories and writes the dataset configuration YAML.

In [ ]:
import os
import yaml
from pathlib import Path

# Define workspace paths
DATASET_DIR = Path('/content/datasets/dentex_seg')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

data_yaml = {
    'path': str(DATASET_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'impacted_tooth',
        1: 'caries',
        2: 'periapical_lesion',
        3: 'deep_caries'
    },
    'nc': 4,
    'description': 'DENTEX 4-Class abnormal-tooth whole contour instance segmentation'
}

yaml_path = DATASET_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f'Created dataset config at: {yaml_path}')
print(open(yaml_path).read())

## 3. Training: Transfer Fine-Tuning YOLOv8n-Seg
Hyperparameters:
- Base checkpoint: `yolov8n-seg.pt` (COCO pre-trained)
- Image resolution: `imgsz=640`
- Batch size: `batch=8` (T4 friendly, deterministic)
- Epochs: `epochs=30` with patience=10
- Optimizer: `AdamW` (lr0=0.001, lrf=0.01, cos_lr=True)
- Seed: `seed=42` for strict reproducibility

In [ ]:
from ultralytics import YOLO

# Load pre-trained nano segmentation model
model = YOLO('yolov8n-seg.pt')

# Train on DENTEX dataset
results = model.train(
    data=str(DATASET_DIR / 'data.yaml'),
    epochs=30,
    batch=8,
    imgsz=640,
    device=0,
    seed=42,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    patience=10,
    save=True,
    project='/content/runs/segment',
    name='dentex_yolov8n_seg'
)

print('Training complete! Best weights saved to:', model.trainer.best)

## 4. Evaluation on Validation Split
Computes precision, recall, F1, Box mAP50, and Mask mAP50 across all 4 dental pathology classes.

In [ ]:
# Validate best checkpoint
best_pt = '/content/runs/segment/dentex_yolov8n_seg/weights/best.pt'
eval_model = YOLO(best_pt)
metrics = eval_model.val(data=str(DATASET_DIR / 'data.yaml'), split='val', imgsz=640, device=0)

print('--- VALIDATION RESULTS ---')
print(f'Box mAP50:  {metrics.box.map50:.4f}')
print(f'Box mAP50-95: {metrics.box.map:.4f}')
print(f'Mask mAP50: {metrics.seg.map50:.4f}')
print(f'Mask mAP50-95: {metrics.seg.map:.4f}')
for i, c in enumerate(metrics.names.values()):
    print(f'Class {c:18s} | Box mAP50: {metrics.box.maps[i]:.4f} | Mask mAP50: {metrics.seg.maps[i]:.4f}')

## 5. Export to ONNX (DENTRACE Production Format)
Exports the trained PyTorch checkpoint to ONNX format with opset 12 for high-speed local inference.

In [ ]:
import hashlib

# Export to ONNX
onnx_path = eval_model.export(
    format='onnx',
    opset=12,
    simplify=True,
    dynamic=False,
    imgsz=640
)

# Compute SHA256 checksum
sha = hashlib.sha256()
with open(onnx_path, 'rb') as f:
    while chunk := f.read(1024 * 1024):
        sha.update(chunk)

print(f'Exported ONNX model: {onnx_path}')
print(f'SHA256: {sha.hexdigest()}')
print('Copy this ONNX file to models/checkpoints/dentex_yolov8n_seg_best.onnx in the DENTRACE repository.')